<a href="https://colab.research.google.com/github/MarcinMarud/flyrank_internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MarcinMarud/flyrank_internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row = one piece of content on one specific day. So if page has 90 days of data, that means 90 rows one per day. Data comes from fact_content_daily_performance, and dim_content

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{HF_TOKEN}')")

rel = "hf://datasets/FlyRank/internship-warehouse"

dim_content_path = f"{rel}/dim_content.parquet"
dim_clients_path = f"{rel}/dim_clients.parquet"
fact_path = f"{rel}/fact_content_daily_performance/month=2026-03/*.parquet"

print("dim_content_path:", dim_content_path)

dim_content_path: hf://datasets/FlyRank/internship-warehouse/dim_content.parquet


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Query 1 checking the grain is actually what I think it is:

I grouped by content id and date to check if any group had more than 1 row

Query 2 how big is this slice:

March gave me [X] rows total, running from [date] to [date]. This is what you would expect from one month based on the users active

Query 3 checking availability:

Out of [X] rows, only [Y] actually have ga4_data_available IS TRUE. So lots of customers dont have ga4, we can't assume its because its zero, we should think about that they did not connect it yet

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

q1 = con.sql(f"""
    SELECT content_hash_id, report_date, COUNT(*) AS n
    FROM read_parquet('{fact_path}')
    GROUP BY 1,2
    HAVING COUNT(*) > 1
    LIMIT 10
""").df()

q2 = con.sql(f"""
    SELECT COUNT(*) AS row_count, MIN(report_date) AS min_date, MAX(report_date) AS max_date
    FROM read_parquet('{fact_path}')
""").df()

q3 = con.sql(f"""
    SELECT COUNT(*) AS total_rows,
           SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_available_rows
    FROM read_parquet('{fact_path}')
""").df()

features = con.sql(f"""
    SELECT
        f.content_hash_id,
        f.report_date,
        f.gsc_impressions AS impressions_to_date,
        f.gsc_clicks AS clicks_to_date,
        f.gsc_avg_position AS avg_position,
        c.word_count,
        DATE_DIFF('day', c.content_created_date, f.report_date) AS content_age_days
    FROM read_parquet('{fact_path}') f
    JOIN read_parquet('{dim_content_path}') c USING (content_hash_id)
""").df()

q1
q2
q3
features.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,content_hash_id,report_date,impressions_to_date,clicks_to_date,avg_position,word_count,content_age_days
0,content_d0dff76c889de68f,2026-03-01,0,0,NaN,2999,17
1,content_67741cce996cfafa,2026-03-01,0,0,NaN,3057,17
2,content_2e6360ad20fd7107,2026-03-01,4,0,3.500,2855,17
3,content_ac8663da7484669a,2026-03-01,8,0,5.875,3281,17
4,content_65c50dfe9d87a585,2026-03-01,0,0,NaN,2779,17


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

It is just for one month, so we could not tell if this is normal or was it only in this one month

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

features_clean = features.dropna(
    subset=["impressions_to_date", "clicks_to_date", "avg_position", "word_count", "content_age_days"]
).copy()

features_clean["target"] = (features_clean["clicks_to_date"] > features_clean["clicks_to_date"].median()).astype(int)

X_honest = features_clean[["impressions_to_date", "avg_position", "word_count", "content_age_days"]]
y = features_clean["target"]

X_train, X_test, y_train, y_test = train_test_split(X_honest, y, test_size=0.3, random_state=42)
model = LogisticRegression(max_iter=1000).fit(X_train, y_train)
honest_score = roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])
print("Honest AUC:", honest_score)

features_clean["leaky_column"] = features_clean["clicks_to_date"] / (features_clean["impressions_to_date"] + 1)

X_leaky = features_clean[["impressions_to_date", "avg_position", "word_count", "content_age_days", "leaky_column"]]
X_train, X_test, y_train, y_test = train_test_split(X_leaky, y, test_size=0.3, random_state=42)
model_leaky = LogisticRegression(max_iter=1000).fit(X_train, y_train)
leaky_score = roc_auc_score(y_test, model_leaky.predict_proba(X_test)[:, 1])
print("Leaky AUC:", leaky_score)

features_clean = features_clean.drop(columns=["leaky_column"])
print("Final honest score to report:", honest_score)


Honest AUC: 0.8277512916132428
Leaky AUC: 0.9956512708092174
Final honest score to report: 0.8277512916132428


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.